# Smart Portfolio Rebalancing — v2

| Step | Description |
|------|-------------|
| 1 | **Setup** — imports, configuration |
| 2 | **Universe** — ticker sampling, data download |
| 3 | **Portfolio config** — selection strategy comparison |
| 4 | **Signals** — technical indicators + strategy evaluation |
| 5 | **Optimisation** — walk-forward weight computation |
| 6 | **Performance** — charts, attribution, ML alpha |

All functions live in **`sprv2.py`** — this notebook is the clean execution layer.

---
## 1  ·  Setup

In [ ]:
import os
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

# ── Make sure sprv2.py is importable ──────────────────────────────────────────
DIR = os.getcwd()
if DIR not in sys.path:
    sys.path.insert(0, DIR)

import sprv2 as sp

print(f'Working directory : {DIR}')
print(f'sprv2 loaded      : {sp.__file__}')

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CONFIGURATION  —  edit these values to customise the run
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# Data
INTERVAL            = '1mo'          # '1mo' | '1wk'
PERIOD              = '16Y'
SP_CSV              = DIR + '/csv/SPfull.csv'

# Universe
SAMPLE_PER_INDUSTRY = 4
EXCLUDE_INDUSTRIES  = ['Energy', 'Real Estate']
ONLY_INDUSTRIES     = []             # [] = all
TICKERS_REMOVE      = ['SAN.PA', 'CA.PA', 'AC.PA', 'ACA.PA', 'TSLA']

# Portfolio config
NOMBRE_TICKERS      = 15             # tickers per selection strategy
PERIOD_TO_START     = '2019-09-26'

# Rolling windows
NB_ROLLING          = 52 if INTERVAL == '1wk' else 12
COV_MA_PERIOD       = 8 * (4 if INTERVAL == '1mo' else 1)

# Signal evaluation
TEST_SIZE           = 0.55          # in-sample fraction
BEST_SIGNAL         = 'default'     # 'default' or int 0-16
SIGNAL_CRITERIA     = 'sharpe'      # 'sharpe' | 'return' | 'sd'

# Optimisation
OPT_OBJECTIVE       = 'sortino'     # 'sortino' | 'sharpe' | 'variance' | 'risk_parity'
MIN_POS             = 3
MIN_W               = 0.04725
MAX_W_RATIO         = 2.86
CORR_CONSTRAINT     = True
MAX_CORR            = 0.4

print('Configuration loaded ✓')

---
## 2  ·  Universe & Data

In [ ]:
df, market_df, ten_year, df_return, tickers, industries = sp.load_universe(
    sp_csv              = SP_CSV,
    interval            = INTERVAL,
    period              = PERIOD,
    sample_per_industry = SAMPLE_PER_INDUSTRY,
    exclude_industries  = EXCLUDE_INDUSTRIES,
    only_industries     = ONLY_INDUSTRIES,
    elements_to_remove  = TICKERS_REMOVE,
)

print(f'\nUniverse  : {df["Close"].shape[1]} tickers')
print(f'History   : {df.index[0].date()} → {df.index[-1].date()}  ({len(df)} periods)')
print(f'Industries: {industries["Industry"].nunique()}')
df['Close'].tail(3)

In [ ]:
# Industry snapshot: correlation heatmap + cumulative return per sector
sp.plot_industry_overview(df, market_df, industries, period='2020')

---
## 3  ·  Portfolio Configuration

Compare 10 ticker-selection strategies (by correlation, volatility, volume, price delta, beta)  
and pick the subset with the best equal-weight return.

In [ ]:
tickers_to_see, tts, list_tickers_to_see = sp.plot_ticker_configs(
    df             = df,
    df_return      = df_return,
    market_df      = market_df,
    tickers_to_see = df['Close'].columns,   # full universe for the scatter
    nombre_tickers = NOMBRE_TICKERS,
    period_to_start= PERIOD_TO_START,
    nb_rolling     = NB_ROLLING,
    list_tts       = 'default',             # 'default' = highest EW return
)

print(f'Selected strategy : {tts}  →  {sp.TTS_MAPPING[tts]}')
print(f'Tickers           : {list(tickers_to_see)}')

In [ ]:
# Correlation heatmap + rolling volatility ranking for the selected universe
sp.plot_ticker_detail(
    df             = df,
    industries     = industries,
    tickers_to_see = tickers_to_see,
    tts_text       = sp.TTS_MAPPING[tts],
    nb_rolling     = NB_ROLLING,
)

---
## 4  ·  Signals & Strategy Selection

In [ ]:
# Compute all technical indicators for every ticker
df_signal = sp.compute_signals(
    df             = df,
    market_df      = market_df,
    tickers_to_see = tickers_to_see,
    nb_rolling     = NB_ROLLING,
)

In [ ]:
# Evaluate all 17 strategies — returns the best one and the end of the test window
best_signal_set = sp.find_best_signal(
    df_signal      = df_signal,
    tickers_to_see = tickers_to_see,
    period_to_start= PERIOD_TO_START,
    nombre_tickers = NOMBRE_TICKERS,
    nb_rolling     = NB_ROLLING,
    best           = BEST_SIGNAL,
    criteria       = SIGNAL_CRITERIA,
    test_size      = TEST_SIZE,
    trace          = True,
)
print(f'\nBest strategy  : [{best_signal_set[0]}] {sp.STRATEGIES[best_signal_set[0]]}')
print(f'End of IS test : {best_signal_set[1]}')

In [ ]:
# Build rebalancing table — one row per period, columns = active positions
rebalancing_table = sp.create_rebalancing_table(
    df_signal      = df_signal,
    tickers_to_see = tickers_to_see,
    best_signal    = best_signal_set[0],
    period_to_start= PERIOD_TO_START,
)
rebalancing_table.tail(5)

In [ ]:
# Interactive per-ticker cumulative return chart (Plotly, log scale)
df_transaction = sp.plot_signal_returns(
    df_signal          = df_signal,
    df                 = df,
    df_return          = df_return,
    industries         = industries,
    tickers_to_see     = tickers_to_see,
    list_tickers_to_see= list_tickers_to_see,
    best_signal_set    = best_signal_set,
    period_to_start    = PERIOD_TO_START,
)

---
## 5  ·  Portfolio Optimisation

In [ ]:
weights_df, equal_w_df = sp.run_optimization(
    df              = df,
    df_return       = df_return,
    market_df       = market_df,
    rebalancing_table= rebalancing_table,
    tickers_to_see  = tickers_to_see,
    period_to_start = PERIOD_TO_START,
    min_pos         = MIN_POS,
    min_w           = MIN_W,
    max_w_ratio     = MAX_W_RATIO,
    cov_ma_period   = COV_MA_PERIOD,
    nb_rolling      = NB_ROLLING,
    objective       = OPT_OBJECTIVE,
    corr_constraint = CORR_CONSTRAINT,
    max_corr        = MAX_CORR,
)

In [ ]:
# Latest 3 weight snapshots (pie charts)
import matplotlib.patches as patches

fig, axes = plt.subplots(1, 3, figsize=(16.5, 5.5))
pct_chg   = df_return.shift(-1)[-3:] * weights_df.iloc[-3]
fig.suptitle(
    'Weights  {}  →  {}'.format(
        pct_chg.index[0].strftime('%Y-%m-%d'),
        pct_chg.index[-1].strftime('%Y-%m-%d')
    ), size=14
)

for t, ax in zip(range(-3, 0), axes):
    last_w = weights_df.iloc[t]
    last_w[last_w > 0.001].plot.pie(
        autopct='%.2f%%', shadow=True, normalize=True,
        explode=[0.075] * last_w[last_w > 0.001].count(), ax=ax,
    )
    ax.set_title(f'Weights  {last_w.name.date()}')
    ax.set_ylabel('')
    if t != -1:
        amount = pct_chg.iloc[t].sum()
        sq = patches.FancyBboxPatch(
            (-0.23, -0.125), 0.46, 0.25,
            boxstyle='round,pad=0.02',
            lw=1, edgecolor='black', facecolor='black',
            transform=ax.transData,
        )
        ax.add_patch(sq)
        ax.annotate(f'{amount:.2%}', (0, 0), fontsize=13, color='white',
                    ha='center', va='center')
plt.tight_layout()
plt.show()

# Weight table
weights_df.applymap(lambda x: f'{x:.2%}').iloc[-5:]

---
## 6  ·  Performance Analysis

In [ ]:
# Compute return series needed for downstream charts
returns            = (df_return.loc[PERIOD_TO_START:].shift(-1) * weights_df).sum(axis=1)
returns_benchmark  = market_df[PERIOD_TO_START:]['Chg'].shift(-1).fillna(0)

# Derived bounds  (used in title)
n_pos     = rebalancing_table[PERIOD_TO_START:].count(axis=1).replace(0, np.nan).mean()
max_w_val = sp._max_w_bound(int(n_pos), MAX_W_RATIO)

In [ ]:
# Main performance chart
sp.plot_portfolio_performance(
    returns          = returns,
    returns_benchmark= returns_benchmark,
    ten_year         = ten_year,
    weights_df       = weights_df,
    equal_w_df       = equal_w_df,
    df_return        = df_return,
    rebalancing_table= rebalancing_table,
    tickers_to_see   = tickers_to_see,
    period_to_start  = PERIOD_TO_START,
    nb_rolling       = NB_ROLLING,
    min_pos          = MIN_POS,
    min_w            = MIN_W,
    max_w_val        = max_w_val,
    cov_ma_period    = COV_MA_PERIOD,
    best_signal_id   = best_signal_set[0],
)

In [ ]:
# Return & volatility scatter per share
sp.plot_returns_per_share(weights_df, df_return, tickers_to_see, NB_ROLLING)

In [ ]:
# Per-share and per-industry cumulative return
sp.plot_returns_share_industry(weights_df, df_return, tickers_to_see, industries)

In [ ]:
# 3D return surface: allocation × time
sp.market_plot_3d(returns, returns_benchmark)

In [ ]:
# Quick summary table
avg_pos   = rebalancing_table[PERIOD_TO_START:].count(axis=1).mean()
reb_ret   = (1 + (df_return.loc[PERIOD_TO_START:].shift(-1) * weights_df).sum(axis=1)).prod()
eqw_ret   = (1 + (equal_w_df * df_return[PERIOD_TO_START:].shift(-1)).sum(axis=1)).prod()
mkt_ret   = (1 + returns_benchmark).prod()
ini_ret   = (1 + (1/df_return.shape[1]) * df_return.loc[PERIOD_TO_START:].shift(-1).sum(axis=1)).prod()

summary = pd.DataFrame({
    'Strategy':            ['Rebalanced', 'Equal-w + Signal', 'Equal-w', 'Benchmark'],
    'Cumulative Return':   [f'{x:.2f}' for x in [reb_ret, eqw_ret, ini_ret, mkt_ret]],
    'Avg Positions':       [f'{avg_pos:.1f}', '—', '—', '—'],
    'Periods':             [len(weights_df)] * 4,
    'Tickers':             [len(tickers_to_see)] * 4,
}).set_index('Strategy')

display(summary)

---
## 7  ·  ML Alpha  *(optional)*

Cross-sectional alpha signals evaluated with XGBoost and `analyze_alpha_signal`.

In [ ]:
# Build raw alpha features
features = sp.build_ml_features(df_signal)

for name, alpha in features.items():
    sp.analyze_alpha_signal(
        positions  = alpha,
        df_return  = df_return,
        title      = f'Alpha: {name}',
        nb_rolling = NB_ROLLING,
        lags       = [0, 1, 2],
        short      = False,
    )

In [ ]:
# XGBoost predictions (requires xgboost)
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor, plot_importance

window = 6
label  = df_return.rolling(window, 1).mean().stack()

pos_features = {
    name: sp.alpha_to_positions(alpha).shift(window).stack().reindex(label.index)
    for name, alpha in features.items()
}
X = pd.concat(pos_features, axis=1).dropna()
y = label.reindex(X.index)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.5, random_state=11, shuffle=False
)

hyperparams = {
    'colsample_bytree': 0.4, 'learning_rate': 0.00045,
    'max_depth': 6,          'min_child_weight': 20,
    'n_estimators': 1250,    'random_state': 1,
    'reg_lambda': 1,         'subsample': 0.65,
}
model = XGBRegressor(**hyperparams).fit(X_train, y_train)
plot_importance(model, max_num_features=len(features));

In [ ]:
# Out-of-sample predictions  →  alpha signal
predict_X   = pd.concat({n: sp.alpha_to_positions(a) for n, a in features.items()}, axis=1)\
                .stack().loc['2017':].dropna()
predictions = pd.Series(model.predict(predict_X), index=predict_X.index).unstack()

sp.analyze_alpha_signal(
    positions  = predictions,
    df_return  = df_return,
    title      = 'XGBoost Predictions',
    nb_rolling = NB_ROLLING,
    lags       = [0, 1, 2],
)